# 05-1 · The closed-loop mediator (C2) and the veto token (C5)

| | |
|---|---|
| **What it does** | reweights the group's fairness objective from plain-language reactions, then re-picks; a veto token can remove the current pick outright |
| **Proves** | C2 — a closed loop exists, not a one-shot ranker; C5 — the veto arm is real code, not a plan |
| **Ported verbatim from** | Vihaan, `mediator_loop_veto_token.py`, branch `research-tasks`, commit `f045279` "Add files via upload" (2026-07-17). Recoverable at that SHA even after the branch is removed: `git show f045279:mediator_loop_veto_token.py` |
| **Guardrail** | code performs the reweight arithmetic; nothing here asks a model to compute a number |
| **Eval surface** | the same group, run twice — with and without a veto — to see the round count change |

No API key needed for this notebook. The reaction labels below are typed
directly rather than parsed from free text — that parsing step is a genuinely
open task, tracked separately (see "What is not here yet", at the end).


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## The class, read directly — this is Vihaan's code, unmodified

```python
class Mediator:
    def calculate_score(self, option):
        # weighted mean minus a fairness penalty on the spread
        weighted_mean = sum(w_i * s_i)
        return weighted_mean - self.lambda_fairness * variance(scores)

    def update_weights(self, reactions):
        # w_i <- w_i * exp(eta * g(reaction_i)) / Z
        g = {"accept": 0, "reject": 0.2, "strong_dislike": 0.5, "veto": 1.5}[reaction]
        ...

    def apply_veto(self, choice, reactions, candidates):
        # each user has ONE veto token; spending it removes the current pick
        ...

    def run_round(self, candidates, reactions):
        # propose -> veto (if any) -> re-propose if removed -> reweight
        ...
```
`nbio.Mediator` (this repo) — ported with types added, logic unchanged.


## Set up a 3-person group with a real disagreement


In [3]:
# Step 1: three diners, two who like the pick and one who does not
users = ["ann", "ben", "cara"]
satisfaction = {
    "chilli_house": {"ann": 0.95, "ben": 0.90, "cara": 0.10},
    "umami_den":    {"ann": 0.30, "ben": 0.45, "cara": 0.92},
    "sweet_stop":   {"ann": 0.20, "ben": 0.15, "cara": 0.20},
}
mediator = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
print("starting weights:", mediator.weights)
print("starting veto tokens:", mediator.veto_tokens)


starting weights: {'ann': 0.3333333333333333, 'ben': 0.3333333333333333, 'cara': 0.3333333333333333}
starting veto tokens: {'ann': 1, 'ben': 1, 'cara': 1}


## Round 1 — propose, and Cara objects


In [4]:
# Step 2: the group's first recommendation, and Cara's reaction
candidates = list(satisfaction.keys())
choice, scores = mediator.recommend(candidates)
print("proposed:", choice, "| raw scores:", {k: round(v, 3) for k, v in scores.items()})

reactions = {"ann": "accept", "ben": "accept", "cara": "strong_dislike"}
before_weights = dict(mediator.weights)
final = mediator.run_round(candidates, reactions)
nbio.io(1, "run_round (no veto)", {"weights": before_weights},
        {"weights": mediator.weights, "pick": final}, reads=["candidates", "reactions"])
print("round 1 final pick:", final)


proposed: chilli_house | raw scores: {'chilli_house': 0.574, 'umami_den': 0.522, 'sweet_stop': 0.183}
+- STEP 1  run_round (no veto)                                      [py]
| IN      : candidates, reactions
| NEW     : pick
| CHANGED : weights
+-------------------------------------------------------------------------
round 1 final pick: chilli_house


### Reading it — the objection changes the weights, and the numbers are auditable

Cara's `strong_dislike` (gradient 0.5) raised her weight relative to Ann and
Ben's `accept` (gradient 0.0). The exponentiated-gradient update is the same
arithmetic whether the reaction came from a typed label (as here) or a parsed
one — the parser is the only piece not built yet.


## Round 2 — Cara spends her veto


In [5]:
# Step 3: same candidates, but this time Cara vetoes outright
mediator2 = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
choice, _ = mediator2.recommend(candidates)
print("proposed:", choice)

reactions_veto = {"ann": "accept", "ben": "accept", "cara": "veto"}
before_tokens = dict(mediator2.veto_tokens)
final_after_veto = mediator2.run_round(list(candidates), reactions_veto)
print("Cara's veto tokens: before =", before_tokens, " after =", mediator2.veto_tokens)
print("round 1 final pick (with veto):", final_after_veto)
assert final_after_veto != choice, "the veto should have removed the original pick"
print("PASS -- vetoing the proposed option forced a different pick")


proposed: chilli_house
Cara's veto tokens: before = {'ann': 1, 'ben': 1, 'cara': 1}  after = {'ann': 1, 'ben': 1, 'cara': 0}
round 1 final pick (with veto): umami_den
PASS -- vetoing the proposed option forced a different pick


## Run several rounds — does it reach consensus?


In [6]:
# Step 4: repeat until nbio.consensus_reached() says the worst-off member is satisfied
mediator3 = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
candidates3 = list(satisfaction.keys())
trace = []
for round_no in range(1, 6):
    choice, _ = mediator3.recommend(candidates3)
    worst_score = min(satisfaction[choice][u] for u in users)
    reaction = "accept" if worst_score >= 0.6 else "strong_dislike"
    worst_user = min(users, key=lambda u: satisfaction[choice][u])
    reactions_round = {u: ("accept" if u != worst_user else reaction) for u in users}
    trace.append({"round": round_no, "pick": choice, "worst_user": worst_user,
                  "worst_score": round(worst_score, 3), "consensus": nbio.consensus_reached(worst_score)})
    if nbio.consensus_reached(worst_score):
        break
    mediator3.run_round(candidates3, reactions_round)

nbio.table(trace, cols=["round", "pick", "worst_user", "worst_score", "consensus"])


round  pick          worst_user  worst_score  consensus
-------------------------------------------------------
1      chilli_house  cara        0.100        False    
2      umami_den     ann         0.300        False    
3      chilli_house  cara        0.100        False    
4      umami_den     ann         0.300        False    
5      umami_den     ann         0.300        False    


### Reading it — an honest surprise

The trace above does NOT converge in 5 rounds: the pick oscillates between
`chilli_house` and `umami_den` because whichever user is worst-off flips each
round, and reweighting the previous round's worst-off user is what makes the
NEXT round's winner change again. This is a real property of Vihaan's reweight
rule as written, not a bug introduced here — `eta=0.5` moves weight fast enough
to overshoot. It is exactly the kind of thing an oscillation/damping check
(smaller eta, weight decay, or a round cap with a fallback) needs to catch
before this loop is trusted to actually reach agreement. Tracked as an open
question for **PRD-5**'s harness: report the fraction of synthetic groups that
fail to converge within the round cap, not only the ones that do.

### What is not here yet

This notebook proves the reweight loop and the veto mechanic are real, working
code, including a real failure mode (non-convergence) worth reporting rather
than hiding. It does NOT yet include: (1) parsing a free-text reaction into one
of the four reaction labels above — currently the labels are typed directly —
and (2) wiring this loop into the actual chat graph so a real conversation
drives it. Both are open tasks; see `research/01_fair-mediator/prds/PRD_5` for
the harness that would exercise this at scale, and `PRD_3` for the notebook
that would add the objection parser.

Next: **06-evaluate** would sweep this loop over many synthetic groups — not yet
built.
